In [6]:
import numpy as np
from bvh import Bvh

def rotate_bvh(input_bvh, output_bvh, rotation_deg=90, axis='y'):
    with open(input_bvh, 'r') as f:
        bvh = Bvh(f.read())

    rotation_rad = np.radians(rotation_deg)

    if axis == 'y':
        R = np.array([
            [np.cos(rotation_rad), 0, np.sin(rotation_rad)],
            [0, 1, 0],
            [-np.sin(rotation_rad), 0, np.cos(rotation_rad)]
        ])
    elif axis == 'x':
        R = np.array([
            [1, 0, 0],
            [0, np.cos(rotation_rad), -np.sin(rotation_rad)],
            [0, np.sin(rotation_rad), np.cos(rotation_rad)]
        ])
    elif axis == 'z':
        R = np.array([
            [np.cos(rotation_rad), -np.sin(rotation_rad), 0],
            [np.sin(rotation_rad), np.cos(rotation_rad), 0],
            [0, 0, 1]
        ])
    else:
        raise ValueError("Axis must be one of ['x', 'y', 'z'].")

    # ✅ frames là list các list số (float)
    frames = np.array(bvh.frames, dtype=float)

    # Xoay root translation (3 giá trị đầu tiên mỗi frame)
    root_pos = frames[:, :3]
    frames[:, :3] = root_pos @ R.T

    # Ghi lại file mới
    with open(output_bvh, 'w') as f:
        f.write(str(bvh))
        f.write("\nMOTION\n")
        f.write(f"Frames: {bvh.nframes}\n")
        f.write(f"Frame Time: {bvh.frame_time}\n")
        for frame in frames:
            f.write(" ".join(f"{v:.6f}" for v in frame) + "\n")

    print(f"✅ Rotated BVH saved to {output_bvh}")


In [8]:
rotate_bvh("/home/anhndt/animating_image/external/AnimatedDrawings/examples/bvh/custom/siu.bvh",
           "/home/anhndt/animating_image/external/AnimatedDrawings/examples/bvh/custom/siu.bvh",
           rotation_deg=90,
           axis="y")

✅ Rotated BVH saved to /home/anhndt/animating_image/external/AnimatedDrawings/examples/bvh/custom/siu.bvh


In [2]:
def skip_bvh_frames(input_path: str, output_path: str, step: int = 2):
    """
    Skip frames in a BVH file by keeping only every 'step' frame.
    Adjusts both frame count and frame time.
    """
    hierarchy_lines = []
    motion_lines = []
    in_motion = False

    with open(input_path, "r") as f:
        for line in f:
            if line.strip().startswith("MOTION"):
                in_motion = True
                hierarchy_lines.append(line)
                continue

            if not in_motion:
                hierarchy_lines.append(line)
            else:
                motion_lines.append(line)

    # Extract meta
    original_frames = int(motion_lines[0].split(":")[1].strip())
    old_frame_time = float(motion_lines[1].split(":")[1].strip())

    # Frame data
    frame_data = motion_lines[2:]

    # Skip frames
    new_frame_data = frame_data[::step]
    new_frame_count = len(new_frame_data)

    # Compute new frame time
    new_frame_time = old_frame_time * step

    # Build output
    output_lines = []
    output_lines.extend(hierarchy_lines)
    output_lines.append(f"Frames: {new_frame_count}\n")
    output_lines.append(f"Frame Time: {new_frame_time:.6f}\n")
    output_lines.extend(new_frame_data)

    with open(output_path, "w") as f:
        f.writelines(output_lines)

    return output_path


In [3]:
skip_bvh_frames(
    input_path="/home/anhndt/animating_image/external/AnimatedDrawings/examples/bvh/custom/russian_dancing.bvh",
    output_path="/home/anhndt/animating_image/external/AnimatedDrawings/examples/bvh/custom/russian_dancing_skip_10.bvh",
    step=10  # giữ mỗi 2 frame
)


'/home/anhndt/animating_image/external/AnimatedDrawings/examples/bvh/custom/russian_dancing_skip_10.bvh'

In [1]:
from PIL import Image, ImageSequence
import numpy as np

def align_gif(input_path, output_path, align="bottom", margin=0):
    """
    Aligns the visible (non-transparent) part of a character inside a GIF
    to one edge of the frame (top, bottom, left, right) and outputs a new GIF.

    Parameters
    ----------
    input_path : str
        Path to the input GIF file.
    output_path : str
        Path where the aligned GIF will be saved.
    align : str, optional
        Direction to align the character. Must be one of:
        - "bottom": Place character touching the bottom edge.
        - "top": Place character touching the top edge.
        - "left": Place character touching the left edge.
        - "right": Place character touching the right edge.
        Default is "bottom".
    margin : int, optional
        Additional offset (in pixels) between the character and the aligned edge.
        Positive values move the character farther from the edge.
        Default is 0.

    Returns
    -------
    str
        Path to the output GIF file.

    Notes
    -----
    - Transparent pixels (alpha <= 10) are treated as background.
    - Each frame is processed independently.
    - The output GIF keeps the original duration and looping behavior.
    """

    gif = Image.open(input_path)
    frames = []

    for frame in ImageSequence.Iterator(gif):
        # Convert frame to RGBA to ensure alpha transparency exists
        frame = frame.convert("RGBA")
        arr = np.array(frame)

        # Extract alpha channel and find all visible (non-transparent) pixels
        alpha = arr[:, :, 3]
        ys, xs = np.where(alpha > 10)  # consider alpha > 10 as visible character pixels

        # If the frame is empty (no visible pixels), keep the frame unchanged
        if len(ys) == 0:
            frames.append(frame)
            continue

        height, width = arr.shape[0], arr.shape[1]

        # Compute bounding box of visible character
        top, bottom = ys.min(), ys.max()
        left, right = xs.min(), xs.max()

        # Compute shift offsets for alignment
        shift_x = 0
        shift_y = 0

        if align == "bottom":
            # Move character so its lowest pixel touches bottom edge
            shift_y = (height - 1 - bottom) - margin

        elif align == "top":
            # Move character so its highest pixel touches top edge
            shift_y = -top + margin

        elif align == "left":
            # Move character so its leftmost pixel touches left edge
            shift_x = -left + margin

        elif align == "right":
            # Move character so its rightmost pixel touches right edge
            shift_x = (width - 1 - right) - margin

        # Make sure shifts do not move image out of bounds
        shift_x = int(max(shift_x, -left))
        shift_y = int(max(shift_y, -top))

        # Create a new transparent frame and paste shifted character
        new_frame = Image.new("RGBA", (width, height), (0, 0, 0, 0))
        new_frame.paste(frame, (shift_x, shift_y))

        frames.append(new_frame)

    # Save all frames as an animated GIF
    frames[0].save(
        output_path,
        save_all=True,
        append_images=frames[1:],
        loop=0,
        duration=gif.info.get("duration", 50),
        disposal=2
    )

    return output_path


In [2]:
align_gif(
    input_path="/home/anhndt/animating_image/src/configs/characters/char13/standing.gif",
    output_path="/home/anhndt/animating_image/src/configs/characters/char13/standing_aligned_bottom.gif",
    align="bottom",
    margin=0
    )

'/home/anhndt/animating_image/src/configs/characters/char13/standing_aligned_bottom.gif'

In [5]:
import bpy
import numpy as np
from os import listdir, path

def fbx2bvh(data_path, file):
    sourcepath = data_path+"/"+file
    bvh_path = data_path+"/"+file.split(".fbx")[0]+".bvh"

    bpy.ops.import_scene.fbx(filepath=sourcepath)

    frame_start = 9999
    frame_end = -9999
    action = bpy.data.actions[-1]
    if  action.frame_range[1] > frame_end:
      frame_end = action.frame_range[1]
    if action.frame_range[0] < frame_start:
      frame_start = action.frame_range[0]

    frame_end = np.max([60, frame_end])
    bpy.ops.export_anim.bvh(filepath=bvh_path,
                            frame_start=int(frame_start),
                            frame_end=int(frame_end), root_transform_only=True)
    bpy.data.actions.remove(bpy.data.actions[-1])
    print(data_path+"/"+file+" processed.")


In [6]:
fbx2bvh("/home/anhndt/animating_image/notebook", "siu.fbx")

FBX version: 7400
BVH Exported: /home/anhndt/animating_image/notebook/siu.bvh frames:183

/home/anhndt/animating_image/notebook/siu.fbx processed.
